## Построение пайплайнов для загрузки параметров

### Загрузка всех датасетов

В начале загрузим все обработанные версии датасетов.

In [1]:
!pip install scikit-optimize
!pip install catboost
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 4.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, mean_squared_error, make_scorer
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')


In [3]:
df_adult = pd.read_csv('/content/adult_cleaned_new.csv')
df_bank = pd.read_csv('/content/bank_marketing_cleaned_new.csv')
df_cal = pd.read_csv('/content/california_housing_cleaned_new.csv')
df_sc = pd.read_csv('/content/superconductivity_cleaned_new.csv')
df_sp = pd.read_csv('/content/spambase_cleaned_new.csv')

In [4]:
datasets = {
    'Adult': df_adult,
    'Bank': df_bank,
    'California': df_cal,
    'Superconductivity': df_sc,
    'Spam': df_sp
}

In [5]:
for name, data in datasets.items():
  print(f'{name}:', end = '\n')
  print(len(data))

Adult:
48842
Bank:
45211
California:
20640
Superconductivity:
21263
Spam:
4601


In [6]:
for name, data in datasets.items():
  print(f'{name}:', end = '\n')
  print(data['target'].nunique())

Adult:
2
Bank:
2
California:
3842
Superconductivity:
3007
Spam:
2


### Создаем конфиги

**Конфиги датасетов**

В конфиге задаем:

1. Какой тип задачи решается в датасете
2. Колонку с таргетом
3. Колонки с категориальными и числовыми фичами
4. Флаг того, большой датасет или маленький (в зависимости от этого определяется число фолдов при оценке качества моделей).

In [7]:
DATASET_CONFIGS = {
    'Adult': {
        'dataframe': df_adult,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': ['workclass', 'education', 'marital-status', 'occupation',
                             'relationship', 'race', 'sex', 'native-country'],
        'numeric_cols': ['age', 'fnlwgt', 'education-num', 'capital-gain',
                        'capital-loss', 'hours-per-week'],
        'size': 'large'
    },
    'Bank': {
        'dataframe': df_bank,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': ['job', 'marital', 'education', 'contact',
                            'day_of_week', 'month', 'poutcome'],
        'numeric_cols': ['age', 'balance', 'campaign', 'pdays', 'previous',
                        'default', 'housing', 'loan', 'was_contacted'],
        'size': 'large'
    },
    'California': {
        'dataframe': df_cal,
        'task': 'regression',
        'target_col': 'target',
        'categorical_cols': ['ocean_proximity'],
        'numeric_cols': ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                        'total_bedrooms', 'population', 'households', 'median_income',
                        'rooms_per_household', 'bedrooms_per_room', 'population_per_household'],
        'size': 'large'
    },
    'Superconductivity': {
        'dataframe': df_sc,
        'task': 'regression',
        'target_col': 'target',
        'categorical_cols': [],
        'numeric_cols': None,  # Все колонки кроме target
        'size': 'large'
    },
    'Spam': {
        'dataframe': df_sp,
        'task': 'classification',
        'target_col': 'target',
        'categorical_cols': [],
        'numeric_cols': None,  # Все колонки кроме target
        'size': 'small'
    }
}

**Конфиги моделей**

В конфиге задаем:

1. Какой тип модели используем
2. Словарь с перебираемыми параметрами
3. Список поддерживаемых задач

In [8]:
#Конфиги моделей с перебираемыми параметрами
MODEL_CONFIGS = {
    'LogisticRegression': {
        'class': LogisticRegression,
        'params': {
            'C': Real(0.01, 100, prior='log-uniform'),
            'penalty': Categorical(['l1', 'l2']),
            #'penalty': Categorical(['l1', 'l2']),
            #'solver': Categorical(['liblinear', 'saga']),
            'solver': Categorical(['liblinear']),
            'max_iter': Integer(100, 1000),
            'tol': [1e-3]
        },
        'supports': ['classification']
    },
    'Ridge': {
        'class': Ridge,
        'params': {
            'alpha': Real(0.001, 100, prior='log-uniform'),
            'solver': Categorical(['auto', 'svd', 'cholesky', 'lsqr', 'sag'])
        },
        'supports': ['regression']
    },
    'Lasso': {
        'class': Lasso,
        'params': {
            'alpha': Real(0.0001, 10, prior='log-uniform'),
            'max_iter': Integer(1000, 5000),
            'selection': Categorical(['cyclic', 'random'])
        },
        'supports': ['regression']
    },

    'ElasticNet': {
        'class': ElasticNet,
        'params': {
            'alpha': Real(0.001, 10, prior='log-uniform'),
            'l1_ratio': Real(0.1, 0.9),  # 0 = Ridge, 1 = Lasso
            'max_iter': Integer(1000, 5000)
        },
        'supports': ['regression']
    },
    'RandomForestClassifier': {
        'class': RandomForestClassifier,
        'params': {
            'n_estimators': Integer(50, 300),
            'max_depth': Integer(3, 20),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 10),
            'max_features': Categorical(['sqrt', 'log2', None])
        },
        'supports': ['classification']
    },
    'RandomForestRegressor': {
        'class': RandomForestRegressor,
        'params': {
            #'n_estimators': Integer(50, 500),
            'n_estimators': Integer(50, 300),
            'max_depth': Integer(3, 20),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 10),
            'max_features': Categorical(['sqrt', 'log2', None])
        },
        'supports': ['regression']
    },
    'LightGBMClassifier': {
        'class': lgb.LGBMClassifier,
        'params': {
            'n_estimators': Integer(50, 500),
            'max_depth': Integer(3, 15),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'num_leaves': Integer(10, 100),
            'min_child_samples': Integer(5, 50),
            'subsample': Real(0.6, 1.0),
            'colsample_bytree': Real(0.6, 1.0),
            'verbose': [-1]
        },
        'supports': ['classification']
    },
    'LightGBMRegressor': {
        'class': lgb.LGBMRegressor,
        'params': {
            'n_estimators': Integer(50, 500),
            'max_depth': Integer(3, 15),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'num_leaves': Integer(10, 100),
            'min_child_samples': Integer(5, 50),
            'subsample': Real(0.6, 1.0),
            'colsample_bytree': Real(0.6, 1.0),
            'verbose': [-1]
        },
        'supports': ['regression']
    },
    'CatBoostClassifier': {
        'class': CatBoostClassifier,
        'params': {
            'iterations': Integer(50, 350),
            'depth': Integer(3, 10),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'l2_leaf_reg': Real(1, 10),
            'border_count': Integer(32, 255),
            'verbose': [0]
        },
        'supports': ['classification']
    },
    'CatBoostRegressor': {
        'class': CatBoostRegressor,
        'params': {
            'iterations': Integer(50, 350),
            'depth': Integer(3, 10),
            'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
            'l2_leaf_reg': Real(1, 10),
            'border_count': Integer(32, 255),
            'verbose': [0]
        },
        'supports': ['regression']
    }
}

### Пишем препроцессинг

In [9]:
def get_preprocessor(dataset_name, model_type):
    """
    Создает препроцессор в зависимости от датасета и типа модели.

    Args:
        dataset_name: Название датасета
        model_type: Тип модели ('linear', 'tree', 'boosting')
    Returns:
        ColumnTransformer или None
    """
    config = DATASET_CONFIGS[dataset_name]
    categorical_cols = config['categorical_cols']
    numeric_cols = config['numeric_cols']

    # Если все признаки числовые
    if not categorical_cols and numeric_cols is None:
        if model_type == 'linear':
            return RobustScaler()  # Устойчив к выбросам
        else:
            return None  # Деревьям скейлинг не нужен

    # Если есть категориальные признаки
    if model_type == 'linear':
        # Для линейных моделей: OneHotEncoder + RobustScaler
        return ColumnTransformer([
            ('num', RobustScaler(), numeric_cols if numeric_cols else
             [col for col in config['dataframe'].columns
              if col != config['target_col'] and col not in categorical_cols]),
            ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),
             categorical_cols)
        ])

    elif model_type == 'tree':
        return ColumnTransformer([
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
             categorical_cols)
        ], remainder='passthrough')

    elif model_type == 'boosting':
        # Для LightGBM/CatBoost: ничего не делаем, они сами обработают категории
        return None


def prepare_data(dataset_name):
    """
    Подготавливает X и y для датасета.
    """
    config = DATASET_CONFIGS[dataset_name]
    df = config['dataframe']
    target_col = config['target_col']

    X = df.drop(target_col, axis=1)
    y = df[target_col]

    # Для датасетов без явных числовых колонок берем все колонки
    if config['numeric_cols'] is None:
        config['numeric_cols'] = list(X.select_dtypes(include=[np.number]).columns)

    return X, y

Выберем дополнительно тип кросс-валидации в зависимости от датасета.

Также сделаем маппинг модели и типа задачи, которую она решает, с классом в sklearn.

In [10]:
def get_cv_splitter(dataset_name, task):
    """
    Выбирает тип кросс-валидации в зависимости от размера датасета и задачи.
    """
    config = DATASET_CONFIGS[dataset_name]

    if config['size'] == 'large':
        n_splits = 3
    else:
        n_splits = 3

    if task == 'classification':
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    else:
        return KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [11]:
def get_model_for_task(model_name, task):
    """
    Подбирает правильную модель в зависимости от задачи.
    """
    mapping = {
        # Для классификации
        ('LogisticRegression', 'classification'): 'LogisticRegression',
        ('RandomForest', 'classification'): 'RandomForestClassifier',
        ('LightGBM', 'classification'): 'LightGBMClassifier',
        ('CatBoost', 'classification'): 'CatBoostClassifier',

        # Для регрессии
        #('LogisticRegression', 'regression'): 'Ridge',  # По умолчанию Ridge
        ('Ridge', 'regression'): 'Ridge',
        ('Lasso', 'regression'): 'Lasso',
        ('ElasticNet', 'regression'): 'ElasticNet',
        ('RandomForest', 'regression'): 'RandomForestRegressor',
        ('LightGBM', 'regression'): 'LightGBMRegressor',
        ('CatBoost', 'regression'): 'CatBoostRegressor'
    }

    return mapping.get((model_name, task))


def get_model_type(model_name):
    """
    Определяет тип модели для выбора препроцессора.
    """
    if model_name in ['LogisticRegression', 'Ridge', 'Lasso', 'ElasticNet']:
        return 'linear'
    elif model_name == 'RandomForest':
        return 'tree'
    elif model_name in ['LightGBM', 'CatBoost']:
        return 'boosting'
    else:
        # На всякий случай: если передано полное имя модели
        if 'Logistic' in model_name:
            return 'linear'
        elif 'Ridge' in model_name or 'Lasso' in model_name or 'ElasticNet' in model_name:
            return 'linear'
        elif 'RandomForest' in model_name:
            return 'tree'
        elif 'LightGBM' in model_name or 'CatBoost' in model_name:
            return 'boosting'
        else:
            raise ValueError(f"Неизвестный тип модели: {model_name}")

In [12]:
## Функция для логирования
import csv
import os

def log_best_to_csv(log_file, iteration, best_score, best_params, task, metric_name):
    """Добавляет строку в CSV-файл логов (режим append)."""
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([iteration, best_score, best_params, task, metric_name])

### Основная функция для оптимизации

Напишем функцию, которая принимает на вход:

1. Датасет
2. Модель
3. Число Итераций

И ищет наилучшие параметры модели по байесовскому подходу, используя гауссовский случайный процесс.

In [13]:
def run_catboost_optuna(dataset_name, model_name='CatBoost', n_trials=50,
                        log_interval=5, log_file=None):
    """
    Оптимизация CatBoost с помощью Optuna (TPE).
    """
    if model_name != 'CatBoost':
        raise ValueError("Эта функция только для CatBoost")

    task = DATASET_CONFIGS[dataset_name]['task']
    full_model_name = get_model_for_task(model_name, task)
    model_config = MODEL_CONFIGS[full_model_name]
    model_class = model_config['class']

    print(f"\n{'='*70}")
    print(f"Optuna (CatBoost) оптимизация на {dataset_name}")
    print(f"Задача: {task.upper()}")
    print(f"{'='*70}")

    # Данные
    X, y = prepare_data(dataset_name)
    print(f"Размер данных: X={X.shape}, y={y.shape}")

    # Категориальные признаки
    categorical_cols = DATASET_CONFIGS[dataset_name]['categorical_cols']
    cat_features = None
    if categorical_cols:
        cat_features = [X.columns.get_loc(col) for col in categorical_cols if col in X.columns]
        print(f"Категориальные признаки (индексы): {cat_features}")

    # Метрика и кросс-валидация
    cv = get_cv_splitter(dataset_name, task)
    cv_splits = cv.get_n_splits()
    metric_name = 'ROC-AUC' if task == 'classification' else 'MSE'
    print(f"Кросс-валидация: {cv_splits}-fold")
    print(f"Метрика: {metric_name}")

    # Перезаписываем лог-файл (создаём заголовок)
    if log_file:
        with open(log_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['iteration', 'best_score', 'best_params', 'task', 'metric'])

    # Параметры для Optuna
    param_space = model_config['params']

    def objective(trial):
        # Собираем словарь параметров из пространства поиска
        params = {}
        for pname, pspace in param_space.items():
            if isinstance(pspace, Real):
                params[pname] = trial.suggest_float(
                    pname, pspace.low, pspace.high, log=(pspace.prior == 'log-uniform')
                )
            elif isinstance(pspace, Integer):
                params[pname] = trial.suggest_int(pname, pspace.low, pspace.high)
            elif isinstance(pspace, Categorical):
                params[pname] = trial.suggest_categorical(pname, pspace.categories)
            # фиксированные параметры (например, verbose=0) игнорируются

        # Ручная кросс-валидация (без clone)
        if task == 'classification':
            splitter = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
        else:
            splitter = KFold(n_splits=cv_splits, shuffle=True, random_state=42)

        fold_scores = []
        for train_idx, test_idx in splitter.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # Новый экземпляр на каждом фолде – clone не используется
            model = model_class(
                **params,
                cat_features=cat_features,
                verbose=0,
                random_seed=42
            )
            model.fit(X_train, y_train, silent=True)

            if task == 'classification':
                y_pred = model.predict_proba(X_test)[:, 1]
                score = roc_auc_score(y_test, y_pred)
            else:
                y_pred = model.predict(X_test)
                score = -mean_squared_error(y_test, y_pred)  # neg_MSE

            fold_scores.append(score)

        return np.mean(fold_scores)

    # Создаём study с TPE-сэмплером (байесовская оптимизация)
    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction='maximize', sampler=sampler)

    print(f"\nЗапуск оптимизации ({n_trials} trials)...")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_score = study.best_value
    best_params = study.best_params

    # Логирование лучших результатов каждые log_interval итераций
    if log_file:
        best_so_far = -np.inf
        best_params_so_far = None
        written = set()
        for trial in study.trials:
            trial_num = trial.number + 1
            score = trial.value
            if score > best_so_far:
                best_so_far = score
                best_params_so_far = trial.params
            if trial_num % log_interval == 0 and trial_num not in written:
                log_best_to_csv(log_file, trial_num, best_so_far,
                                best_params_so_far, task, metric_name)
                written.add(trial_num)

    print(f"\n{'='*70}")
    print(f"РЕЗУЛЬТАТЫ:")
    print(f"Лучшее значение метрики: {best_score:.4f}")
    print(f"Лучшие параметры: {best_params}")
    print(f"{'='*70}")

    return {
        'dataset': dataset_name,
        'model': model_name,
        'task': task,
        'best_score': best_score,
        'best_params': best_params,
        'study': study
    }

Возможные модели:

1. LogisticRegression
2. Ridge
3. Lasso
4. ElasticNet
5. RandomForest
6. LightGBM
7. CatBoost

Возможные датасеты:

1. Adult
2. Bank
3. California
4. Superconductivity
5. Spam

Добавим логирование на Гугл Диск

In [14]:
from google.colab import drive
import os

# Монтируем Google Диск
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
# Задаём папку для логов на Диске (можно изменить название)
base_log_dir = '/content/drive/MyDrive/ЦУ Курсы/Метопты/Optuna_logs'
os.makedirs(base_log_dir, exist_ok=True)  # создаём, если не существует

Запустим функцию

In [ ]:
log_interval = 5
log_file = os.path.join(base_log_dir, 'optuna_Catboost_Adult_logs.csv')

test_results = run_catboost_optuna('Adult', 'CatBoost', n_trials = 50,
                                        log_interval = log_interval, log_file = log_file)

[I 2026-06-02 20:10:09,398] A new study created in memory with name: no-name-d755bf96-042b-4899-8b96-789bc42657c1



Optuna (CatBoost) оптимизация на Adult
Задача: CLASSIFICATION
Размер данных: X=(48842, 14), y=(48842,)
Категориальные признаки (индексы): [1, 3, 5, 6, 7, 8, 9, 13]
Кросс-валидация: 3-fold
Метрика: ROC-AUC

Запуск оптимизации (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-02 20:10:31,511] Trial 0 finished with value: 0.9260382075037059 and parameters: {'iterations': 162, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 6.387926357773329, 'border_count': 66}. Best is trial 0 with value: 0.9260382075037059.
[I 2026-06-02 20:10:35,215] Trial 1 finished with value: 0.9193828217807001 and parameters: {'iterations': 96, 'depth': 3, 'learning_rate': 0.19030368381735815, 'l2_leaf_reg': 6.41003510568888, 'border_count': 190}. Best is trial 0 with value: 0.9260382075037059.
[I 2026-06-02 20:10:44,447] Trial 2 finished with value: 0.9235065145309633 and parameters: {'iterations': 56, 'depth': 10, 'learning_rate': 0.16967533607196555, 'l2_leaf_reg': 2.9110519961044856, 'border_count': 72}. Best is trial 0 with value: 0.9260382075037059.
[I 2026-06-02 20:10:49,800] Trial 3 finished with value: 0.9161052191239528 and parameters: {'iterations': 105, 'depth': 5, 'learning_rate': 0.05958389350068958, 'l2_leaf_reg': 4.887505167779041, 'border_c

### Поиск по одной модели для всех датасетов

In [16]:
def optimize_catboost_all_datasets(n_trials=50, start_num=0, log_interval=5, base_log_dir='logs'):
    """Запускает CatBoost Optuna на всех совместимых датасетах."""
    os.makedirs(base_log_dir, exist_ok=True)
    all_datasets = list(DATASET_CONFIGS.keys())

    for dataset_name in all_datasets[start_num:]:
        task = DATASET_CONFIGS[dataset_name]['task']
        if get_model_for_task('CatBoost', task) is None:
            print(f"Пропускаем {dataset_name} – задача {task} не поддерживается CatBoost")
            continue

        log_filename = f"optuna_CatBoost_{dataset_name}_logs.csv"
        log_filepath = os.path.join(base_log_dir, log_filename)

        print(f"\n====== Запуск CatBoost на {dataset_name} ======")
        run_catboost_optuna(
            dataset_name=dataset_name,
            model_name='CatBoost',
            n_trials=n_trials,
            log_interval=log_interval,
            log_file=log_filepath
        )

In [17]:
optimize_catboost_all_datasets(n_trials=50, start_num = 2, log_interval=5, base_log_dir='/content/drive/MyDrive/ЦУ Курсы/Метопты/Optuna_logs')


====== Запуск CatBoost на California ======

Optuna (CatBoost) оптимизация на California
Задача: REGRESSION
Размер данных: X=(20640, 12), y=(20640,)
Категориальные признаки (индексы): [8]
Кросс-валидация: 3-fold
Метрика: MSE


[I 2026-06-03 08:08:26,692] A new study created in memory with name: no-name-89a90810-8414-4917-96d8-889341979ab0



Запуск оптимизации (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-03 08:08:36,924] Trial 0 finished with value: -2226084123.697537 and parameters: {'iterations': 162, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 6.387926357773329, 'border_count': 66}. Best is trial 0 with value: -2226084123.697537.
[I 2026-06-03 08:08:38,014] Trial 1 finished with value: -2773306586.3289127 and parameters: {'iterations': 96, 'depth': 3, 'learning_rate': 0.19030368381735815, 'l2_leaf_reg': 6.41003510568888, 'border_count': 190}. Best is trial 0 with value: -2226084123.697537.
[I 2026-06-03 08:08:43,378] Trial 2 finished with value: -2404906342.762264 and parameters: {'iterations': 56, 'depth': 10, 'learning_rate': 0.16967533607196555, 'l2_leaf_reg': 2.9110519961044856, 'border_count': 72}. Best is trial 0 with value: -2226084123.697537.
[I 2026-06-03 08:08:44,821] Trial 3 finished with value: -2964400320.292212 and parameters: {'iterations': 105, 'depth': 5, 'learning_rate': 0.05958389350068958, 'l2_leaf_reg': 4.887505167779041, 'border_

...

[I 2026-06-03 08:24:03,550] Trial 0 finished with value: -102.31175940379764 and parameters: {'iterations': 162, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 6.387926357773329, 'border_count': 66}. Best is trial 0 with value: -102.31175940379764.
[I 2026-06-03 08:24:05,922] Trial 1 finished with value: -169.62411918699016 and parameters: {'iterations': 96, 'depth': 3, 'learning_rate': 0.19030368381735815, 'l2_leaf_reg': 6.41003510568888, 'border_count': 190}. Best is trial 0 with value: -102.31175940379764.
[I 2026-06-03 08:24:21,040] Trial 2 finished with value: -116.57864929927779 and parameters: {'iterations': 56, 'depth': 10, 'learning_rate': 0.16967533607196555, 'l2_leaf_reg': 2.9110519961044856, 'border_count': 72}. Best is trial 0 with value: -102.31175940379764.
[I 2026-06-03 08:24:24,446] Trial 3 finished with value: -176.90573264079225 and parameters: {'iterations': 105, 'depth': 5, 'learning_rate': 0.05958389350068958, 'l2_leaf_reg': 4.887505167779041, 'b

[I 2026-06-03 09:29:05,077] A new study created in memory with name: no-name-de8106db-10e3-4d20-87e6-ff57d6c79098


[I 2026-06-03 09:29:04,983] Trial 49 finished with value: -92.69883179477232 and parameters: {'iterations': 288, 'depth': 10, 'learning_rate': 0.09282547660600998, 'l2_leaf_reg': 1.4421805341764231, 'border_count': 143}. Best is trial 34 with value: -90.86629613449689.

РЕЗУЛЬТАТЫ:
Лучшее значение метрики: -90.8663
Лучшие параметры: {'iterations': 264, 'depth': 10, 'learning_rate': 0.1487670474812849, 'l2_leaf_reg': 1.6467635629434025, 'border_count': 165}

====== Запуск CatBoost на Spam ======

Optuna (CatBoost) оптимизация на Spam
Задача: CLASSIFICATION
Размер данных: X=(4601, 57), y=(4601,)
Кросс-валидация: 3-fold
Метрика: ROC-AUC

Запуск оптимизации (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-03 09:29:29,168] Trial 0 finished with value: 0.9878061916566093 and parameters: {'iterations': 162, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 6.387926357773329, 'border_count': 66}. Best is trial 0 with value: 0.9878061916566093.
[I 2026-06-03 09:29:30,261] Trial 1 finished with value: 0.9847759813479672 and parameters: {'iterations': 96, 'depth': 3, 'learning_rate': 0.19030368381735815, 'l2_leaf_reg': 6.41003510568888, 'border_count': 190}. Best is trial 0 with value: 0.9878061916566093.
[I 2026-06-03 09:29:38,530] Trial 2 finished with value: 0.9869105024979755 and parameters: {'iterations': 56, 'depth': 10, 'learning_rate': 0.16967533607196555, 'l2_leaf_reg': 2.9110519961044856, 'border_count': 72}. Best is trial 0 with value: 0.9878061916566093.
[I 2026-06-03 09:29:41,355] Trial 3 finished with value: 0.984584870953201 and parameters: {'iterations': 105, 'depth': 5, 'learning_rate': 0.05958389350068958, 'l2_leaf_reg': 4.887505167779041, 'border_co

### Архив